# Phase 1.1: Data Pipeline - Loading and Preprocessing
This notebook processes the raw Bengaluru Traffic Police enforcement data.

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

# 1. Load CSV with explicit dtype specifications to avoid pandas type inference errors
csv_path = '../../jan_to_may_police_violation_anonymized.csv' # assuming it's in gridlock root
print(f"Loading data from {csv_path}...")

try:
    df = pd.read_csv(csv_path, dtype={'id': str, 'vehicle_number': str, 'vehicle_type': str, 'device_id': str, 'junction_name': str, 'validation_status': str})
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"ERROR: File not found at {csv_path}. Please place the file in the gridlock directory.")
    # Exit or halt here ideally, but continuing for structure setup
    df = pd.DataFrame()


Loading data from ../../jan_to_may_police_violation_anonymized.csv...


Data loaded successfully.


In [2]:
if not df.empty:
    print("DataFrame Info:")
    df.info()
    print("\n\nUnique Values:")
    print(df.nunique())
    print(f"\nActual row count: {len(df)}")


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 298450 entries, 0 to 298449
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            298450 non-null  object 
 1   latitude                      298450 non-null  float64
 2   longitude                     298450 non-null  float64
 3   location                      295409 non-null  object 
 4   vehicle_number                298450 non-null  object 
 5   vehicle_type                  298450 non-null  object 
 6   description                   0 non-null       float64
 7   violation_type                298450 non-null  object 
 8   offence_code                  298450 non-null  object 
 9   created_datetime              298450 non-null  object 
 10  closed_datetime               0 non-null       float64
 11  modified_datetime             298450 non-null  object 
 12  device_id                   

id                              298450
latitude                        177982
longitude                       177378
location                         10942
vehicle_number                  231890
vehicle_type                        22
description                          0
violation_type                     991
offence_code                       991
created_datetime                 94417
closed_datetime                      0
modified_datetime               298450
device_id                         3070
created_by_id                     2666
center_code                         52
police_station                      54
data_sent_to_scita                   2
junction_name                      169
action_taken_timestamp               0
data_sent_to_scita_timestamp     42161
updated_vehicle_number          143133
updated_vehicle_type                22
validation_status                    5
validation_timestamp            170115
dtype: int64

Actual row count: 298450


## 1.2 Parsing JSON and Timezones

In [3]:
def safe_json_load(x):
    if pd.isna(x):
        return []
    try:
        return json.loads(x)
    except:
        return []

if not df.empty:
    df['violation_list'] = df['violation_type'].apply(safe_json_load)
    df['offence_code_list'] = df['offence_code'].apply(safe_json_load)
    
    # 3. Convert timestamps to IST
    df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='mixed', utc=True)
    df['vist_ist'] = df['created_datetime'] + pd.Timedelta(hours=5, minutes=30)
    
    print("Distribution of violation_count per row:")
    df['violation_count'] = df['violation_list'].apply(len)
    print(df['violation_count'].value_counts(normalize=True) * 100)


Distribution of violation_count per row:
violation_count
1     86.560563
2     11.040710
3      1.810689
4      0.399397
5      0.098509
6      0.060312
7      0.017088
8      0.007036
9      0.005026
11     0.000335
12     0.000335
Name: proportion, dtype: float64


## 1.3 Filtering In-Scope Parking Violations

In [4]:
in_scope_violations = [
    'WRONG PARKING', 'NO PARKING', 'PARKING IN A MAIN ROAD', 
    'PARKING ON FOOTPATH', 'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC', 
    'DOUBLE PARKING', 'PARKING NEAR ROAD CROSSING', 
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS', 
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE', 
    'PARKING OTHER THAN BUS STOP', 'H T V PROHIBITED', 
    'AGAINST ONE WAY/NO ENTRY'
]

if not df.empty:
    def has_in_scope(vlist):
        for v in vlist:
            if v in in_scope_violations:
                return True
        return False
        
    initial_count = len(df)
    df['is_parking'] = df['violation_list'].apply(has_in_scope)
    df_filtered = df[df['is_parking']].copy()
    
    print(f"Filtered from {initial_count} to {len(df_filtered)} in-scope parking violation rows.")
    
    # Get primary violation
    def get_primary(vlist):
        for v in vlist:
            if v in in_scope_violations:
                return v
        return vlist[0] if vlist else None
        
    df_filtered['primary_violation'] = df_filtered['violation_list'].apply(get_primary)
    print("\nTop 5 violation types in filtered data:")
    print(df_filtered['primary_violation'].value_counts().head(5))


Filtered from 298450 to 298450 in-scope parking violation rows.

Top 5 violation types in filtered data:
primary_violation
WRONG PARKING                 147936
NO PARKING                    128965
PARKING IN A MAIN ROAD         17113
PARKING ON FOOTPATH             2456
PARKING NEAR ROAD CROSSING       646
Name: count, dtype: int64


## 1.4 Derive Engineered Features

In [5]:
if not df.empty:
    df_filtered['hour_ist'] = df_filtered['vist_ist'].dt.hour
    df_filtered['dow'] = df_filtered['vist_ist'].dt.dayofweek
    df_filtered['is_weekend'] = df_filtered['dow'].isin([5, 6])
    
    df_filtered['has_junction'] = df_filtered['junction_name'] != "No Junction"
    
    # Repeat plate counter
    plate_counts = df_filtered['vehicle_number'].value_counts()
    df_filtered['repeat_plate'] = df_filtered['vehicle_number'].map(plate_counts).fillna(1)
    
    # Device mode mapping
    device_junction_counts = df_filtered.groupby('device_id')['junction_name'].nunique()
    df_filtered['device_mode'] = df_filtered['device_id'].map(lambda x: 'STATIC' if device_junction_counts.get(x, 0) == 1 else 'MOBILE')
    
    print("Device modes:")
    print(df_filtered['device_mode'].value_counts())


Device modes:
device_mode
MOBILE    202102
STATIC     96348
Name: count, dtype: int64


## 1.5 Footprint and Severity Tables

In [6]:
footprint_weights = {
    'SCOOTER': 0.3, 'MOPED': 0.3,
    'MOTOR CYCLE': 0.35,
    'PASSENGER AUTO': 0.5,
    'CAR': 0.7, 'JEEP': 0.7,
    'TAXI': 0.75, 'OMNI BUS (SMALL)': 0.75,
    'LGV': 0.85, 'TEMPO': 0.85, 'GOODS AUTO': 0.85,
    'VAN': 0.9, 'MAXI-CAB': 0.9,
    'LORRY': 1.0, 'TRUCK': 1.0,
    'BUS': 1.0, 'OMNI BUS': 1.0
}

severity_weights = {
    'DOUBLE PARKING': 1.0,
    'PARKING NEAR ROAD CROSSING': 0.95,
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS': 0.95,
    'PARKING ON FOOTPATH': 0.85,
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE': 0.85,
    'PARKING IN A MAIN ROAD': 0.8,
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC': 0.75,
    'H T V PROHIBITED': 0.75,
    'WRONG PARKING': 0.6, 'NO PARKING': 0.6,
    'PARKING OTHER THAN BUS STOP': 0.55,
    'AGAINST ONE WAY/NO ENTRY': 0.7
}

if not df.empty:
    def get_footprint(vtype):
        if pd.isna(vtype): return 0.5
        for k, v in footprint_weights.items():
            if k in vtype: return v
        return 0.5 # default
        
    df_filtered['vehicle_footprint'] = df_filtered['vehicle_type'].apply(get_footprint)
    df_filtered['severity_score'] = df_filtered['primary_violation'].map(severity_weights).fillna(0.5)
    
    # Save the processed data for next notebook
    output_path = 'exports/processed_violations.parquet'
    df_filtered.to_parquet(output_path, index=False)
    print(f"Saved processed data to {output_path}")


Saved processed data to exports/processed_violations.parquet
